In [1]:
#| default_exp core

# aplnb core
> bAsedPL magics and APL output display for Jupyter and IPython


In [2]:
import html, json, re
from importlib.resources import files
from fastcore.utils import *
from basedpl import Session, AplError, symbols
from basedpl.ipython import load_ipython_extension as _load_help, command_help
from IPython.core.completer import context_matcher, SimpleCompletion
from IPython import get_ipython
from IPython.display import display, Javascript, HTML
from IPython.paths import get_ipython_dir


aplnb adds `%apl` and `%%apl` magics to IPython. A `basedpl.Session` owns the APL workspace, native values, errors and interruption. aplnb renders the captured output and loads the language bar.

The line magic returns a native array or function. The cell magic displays APL session output, including implicit expression results. Both use the same session.

In [3]:
from fastcore.test import *
from basedpl import Array
from IPython.utils.capture import capture_output
import numpy as np

## Rendering output

In [4]:
class AplOut(str):
    "APL output text, displayed in the SAX2 font where HTML is available."
    def __repr__(self): return str(self)
    def _repr_html_(self): return f'<pre class="aplnb_out sax2">{html.escape(self.rstrip(chr(10)))}</pre>'


`AplOut` preserves APL spacing. Its HTML representation escapes the output before putting it in a `<pre>` element.

In [5]:
out = AplOut('1 < 2')
assert '&lt;' in out._repr_html_()
out

1 < 2

## The magics

`APLMagic` accepts an existing `basedpl.Session`. Without one, it starts a session on first use. The language bar and SAX2 stylesheet are loaded once for each magic instance.

In [6]:
_css = r"""<style>
@font-face { font-family:'SAX2'; src: local('SAX2'), url('https://cdn.jsdelivr.net/gh/abrudz/SAX2@master/SAX2.ttf') format('truetype') }
.sax2 { font-family:'SAX2',monospace !important; line-height:1.05 !important }
</style>"""

class APLMagic:
    "IPython APL magics sharing a lazily started bAsedPL session."
    def __init__(self, session=None): self.session,self._loaded = session,False


In [7]:
#| export
@patch
def _load(self:APLMagic):
    if self._loaded: return
    js = files('aplnb')
    keyboard = (files('basedpl')/'keyboard.json').read_text()
    display(Javascript(f"{(js/'lb.js').read_text()}({json.dumps(symbols)}, {(js/'input.js').read_text()}, {keyboard})"))
    display(HTML(_css))
    self._loaded = True

In [8]:
#| export
@patch
def apl(self:APLMagic, line, cell=None):
    "Evaluate a line as a native value or display a cell as an APL session."
    if self.session is None: self.session = Session()
    self._load()
    code = line if cell is None else cell.rstrip()
    show = not (cell is not None and code.endswith(';'))
    if not show: code = code[:-1]
    if (info := command_help(self.session, code)) is not None:
        if show: display(info, raw=True)
        return
    output = []
    try:
        result = (self.session.eval if cell is None else self.session.run)(code)
        output = result.output
    except (AplError, KeyboardInterrupt) as e:
        output = getattr(e, 'output', [])
        raise
    finally:
        if show and output: display(AplOut('\n'.join(output)))
    if cell is None: return result.value

Cell output comes from `Session.run`, which includes implicit expression results. Line output comes from `Session.eval`, which captures explicit output and returns native values. A trailing `;` suppresses all cell display without skipping execution. Output produced before an error is displayed before the error is raised.

Tab completes user and system names in APL input. Completion reads the existing session without starting or evaluating it.

In [9]:
#| export
@patch
@context_matcher(identifier='aplnb.names')
def complete(self:APLMagic, context):
    "Complete visible APL names in line and cell magics."
    empty = dict(completions=[])
    if self.session is None: return empty
    before = '\n'.join(context.full_text.split('\n')[:context.cursor_line] + [context.text_until_cursor])
    if m := re.match(r'%%apl[^\S\n]*\n', before): code = before[m.end():]
    elif m := re.match(r'\s*(?:\w+\s*=\s*)?%apl\s+', context.text_until_cursor): code = context.text_until_cursor[m.end():]
    else: return empty
    if any(m.end() == len(code) for m in re.finditer(r"'(?:[^']|'')*(?:'|$)|⍝[^\n]*", code)): return empty
    start = len(code)
    glyphs = {row[0] for row in symbols} - {'•'}
    while start and code[start-1] not in glyphs and (code[start-1].isalnum() or code[start-1] in '_∆⍙•'): start -= 1
    prefix = code[start:]
    if not prefix: return empty
    return dict(completions=[SimpleCompletion(n, type='APL name') for n in self.session.complete(prefix)],
        matched_fragment=prefix, suppress=True)

In [10]:
def create_magic(shell=None, session=None):
    "Register line and cell magics with `shell`, optionally sharing a bAsedPL `session`."
    if shell is None: shell = get_ipython()
    magic = APLMagic(session)
    shell.register_magic_function(magic.apl, 'line_cell', 'apl')
    matchers = shell.Completer.custom_matchers
    matchers[:] = [m for m in matchers if getattr(m, 'matcher_identifier', None) != 'aplnb.names']
    matchers.append(magic.complete)
    _load_help(shell)
    return magic


In [11]:
#| hide
magic = create_magic()
test_is(magic.session, None)
magic.apl('')

Javascript(// Based on Adám Brudzewsky's APL language bar: https://abrudz.github.io/lb
// MIT License, Copyright (c) 2011-2020 Nikolay G. Nikolov and Adam Brudzevski.
// bAsedPL name completion, editor adapters, dark mode and overlay layout by Jeremy Howard.
((symbols, input, keyboard) => {
    const d = document;
    if (d.querySelector('.ngn_lb') || d.querySelector('meta[name=generator][content^=quarto]')) return;

    const {inCode, aplStart, entry, chord} = input(symbols, keyboard);
    const shortcuts = new Map(symbols.map(([glyph, , , , , shortcut]) => [glyph, shortcut]));
    let leftAlt = false, rightAlt = false;

    function textareaRect(t) {
        const mirror = d.createElement('div'), caret = d.createElement('span'), css = getComputedStyle(t), rect = t.getBoundingClientRect();
        for (const p of ['font', 'lineHeight', 'letterSpacing', 'padding', 'border', 'boxSizing', 'width', 'tabSize']) mirror.style[p] = css[p];
        Object.assign(mirror.style, {position: 'fixed

HTML(<style>
@font-face { font-family:'SAX2'; src: local('SAX2'), url('https://cdn.jsdelivr.net/gh/abrudz/SAX2@master/SAX2.ttf') format('truetype') }
.sax2 { font-family:'SAX2',monospace !important; line-height:1.05 !important }
</style>)

The cell magic displays each expression. Assignments remain silent.

In [12]:
%%apl
m←2 3⍴⍳6
m×10

10 20 30
40 50 60

The line magic returns a native `basedpl.Array`. Use `.np` or `.py` for conversion. Comments are parsed by bAsedPL, including `⍝` inside quoted strings.

In [13]:
z = %apl m ⍝ a matrix
test_is(type(z), Array)
np.testing.assert_array_equal(z.np, [[1, 2, 3], [4, 5, 6]])
test_eq(magic.apl("'a⍝b' ⍝ comment").py, 'a⍝b')
z

[[1., 2., 3.], [4., 5., 6.]]

Python values can be bound through the same session. Python integers stay exact.

In [14]:
magic.session(x=np.arange(1, 4))
value = %apl +/x
test_eq(value.py, 6)
test_is(type(value.py), int)
value

Array(6)

bAsedPL's display commands also work in a cell.

In [15]:
%%apl
]Display (1 2)'ab'

┌→───────────┐
│ ┌→──┐ ┌→─┐ │
│ │1 2│ │ab│ │
│ └~──┘ └──┘ │
└∊───────────┘

A suppressed cell still updates the workspace.

In [16]:
%%apl
quiet←7;


In [17]:
#| hide
with capture_output() as cap: magic.apl('', 'quiet←9 ⋄ ⎕←quiet;')
test_eq(len(cap.outputs), 0)
test_eq(magic.apl('quiet').py, 9)
with capture_output() as cap: magic.apl('', 'quiet ⋄ quiet+1')
assert '9' in cap.outputs[0].data['text/html'] and '10' in cap.outputs[0].data['text/html']

## Names and help

Use `]help name` for help and `]help name -source` for source, in either magic. Leading definition comments supply help. Inspection leaves the function uncalled.

In [18]:
%%apl
double←{⍝ Double the argument
⎕←'called' ⋄ ⍵×2}

In [19]:
with capture_output() as cap: magic.apl(']help double')
assert 'Double the argument' in cap.outputs[0].data['text/markdown']
with capture_output() as cap: magic.apl('', ']help double -source')
assert "⎕←'called'" in cap.outputs[0].data['text/markdown']
test_eq(len(cap.outputs), 1)

Completion follows the magic's workspace, including names defined after registration. Quoted text and comments stay literal.

In [20]:
from IPython.core.completer import provisionalcompleter

In [21]:
shell = get_ipython()
for text, expected in [('%apl dou', 'double'), ('v = %apl dou', 'double'), ('%%apl\n1+•sr', '•src')]:
    with provisionalcompleter(): matches = list(shell.Completer.completions(text, len(text)))
    assert expected in [m.text for m in matches]
for text in ["%%apl\n'dou", '%%apl\n⍝ dou', 'ordinary_python']:
    with provisionalcompleter(): matches = list(shell.Completer.completions(text, len(text)))
    assert not any(m.type == 'APL name' for m in matches)

## Errors and session state

Errors retain source locations and leave completed assignments in the workspace. Incomplete input raises a syntax error without replacing the session.

In [22]:
with capture_output() as cap: test_fail(lambda: magic.apl('', '⎕←42 ⋄ 1÷0'), contains='DOMAIN ERROR')
assert '42' in cap.outputs[0].data['text/html']
test_fail(lambda: magic.apl('', '{⍵+1'), contains='SYNTAX ERROR')
test_eq(magic.apl('quiet').py, 9)

Set `magic.session.timeout` to a per-evaluation deadline in seconds. Ctrl-C interrupts the evaluation. Output captured before cooperative cancellation is displayed. The Rust worker thread retains the session after cancellation. Native-library calls and individual BigInt operations can delay cancellation; the thread is never forcibly killed.

## Installation

`%load_ext aplnb` registers the magics in the current kernel. `aplnb_install` adds the extension to the default IPython profile. Registration does not start an APL worker.

In [23]:
def load_ipython_extension(ipython):
    "Register the APL magics when IPython loads this extension."
    create_magic(shell=ipython)

In [24]:
def create_ipython_config():
    "Called by `aplnb_install` to install magic"
    ipython_dir = Path(get_ipython_dir())
    cf = ipython_dir/'profile_default'/'ipython_config.py'
    cf.parent.mkdir(parents=True, exist_ok=True)
    if cf.exists() and 'aplnb' in cf.read_text(): return print('aplnb already installed!')
    with cf.open(mode='a') as f: f.write("\nc.InteractiveShellApp.extensions.append('aplnb')\n\n")
    print(f"Jupyter config updated at {cf}")

## Symbol input

The browser input method uses bAsedPL's symbol names and aliases. It is active in `%%apl` cells, `%apl` lines and Monaco editors configured for APL. Matching prefers exact names, then prefixes, then abbreviations. A unique match is accepted with Tab or a delimiter. Enter accepts it before the notebook's usual newline or execution action. Ambiguous matches remain unchanged and appear in a clickable list.

Strings, comments and pasted text are not expanded. Cursor movement and Escape cancel automatic expansion. The clickable language bar can still insert literal glyphs.

Hold **left Alt/Option** to type glyphs using [bAsedPL's keyboard](https://answerdotai.github.io/basedpl/keyboard.html): Alt-h `←`, Alt-minus `×`, Alt-equals `÷`, Alt-Shift-a `⍶`. These chords insert literal glyphs, including inside APL strings and comments. Right Option keeps its native keyboard behavior. The glyph map is shared with the bAsedPL REPL.

The JavaScript assertions live in `tests/input.js`. Pytest runs them through fastcdp's QUnit runner using bAsedPL's actual symbol catalogue. Editor integration checks exercise CodeMirror, Monaco and textarea. Install the `dev` extra and Chrome for Testing (`fastcdp-setup --install stable`), then run `pytest -q`. QUnit and the editor fixtures load from CDNs. Node is not required.

## Cleanup

In [25]:
magic.session.close()

In [ ]:
#|hide
#|eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()